```{contents}
```

## Batch Normalization (BN)


Training deep neural networks is difficult because the distribution of activations in intermediate layers **keeps changing** as parameters update.
This phenomenon is known as **internal covariate shift** and leads to:

* unstable gradients
* slow convergence
* sensitivity to initialization
* difficulty using high learning rates

**Batch Normalization** stabilizes training by forcing each layer’s inputs to follow a controlled distribution during training.

> BN makes optimization easier by turning a deep network into a stack of well-conditioned subproblems.

---

### Core Idea

For each mini-batch and for each feature channel:

1. **Normalize** activations to zero mean and unit variance
2. **Re-scale and shift** using learnable parameters

This allows the network to learn the optimal activation distribution while enjoying the optimization benefits of normalization.

---

### Mathematical Formulation

Given a mini-batch $B = {x_1, ..., x_m}$:

**Batch statistics**

$$
\mu_B = \frac{1}{m}\sum_{i=1}^{m} x_i
$$
$$
\sigma_B^2 = \frac{1}{m}\sum_{i=1}^{m}(x_i - \mu_B)^2
$$

**Normalization**

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
$$

**Learnable transformation**

$$
y_i = \gamma \hat{x}_i + \beta
$$

Where:

* $\gamma$ (scale) and $\beta$ (shift) are trainable
* $\epsilon$ prevents numerical instability

---

### Training vs Inference Behavior

| Phase     | Statistics Used                    |
| --------- | ---------------------------------- |
| Training  | mini-batch mean & variance         |
| Inference | running population mean & variance |

---

### Workflow in a Neural Network

```
Linear → BatchNorm → Activation
```

**Why before activation?**
Normalization is applied on raw linear responses to stabilize the nonlinearity that follows.

---

### Optimization Benefits

| Effect                | Explanation                           |
| --------------------- | ------------------------------------- |
| Higher learning rates | gradients become stable               |
| Faster convergence    | smoother loss surface                 |
| Regularization        | mini-batch noise acts as regularizer  |
| Reduced sensitivity   | to initialization and hyperparameters |

---

### Remediation of Common Training Problems

| Problem                         | BN Effect                    |
| ------------------------------- | ---------------------------- |
| Vanishing / exploding gradients | prevents extreme activations |
| Slow convergence                | accelerates optimization     |
| Overfitting                     | mild regularization          |
| Poor initialization             | more forgiving training      |

---

### PyTorch Demonstration

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class BNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

model = BNNet()
```

---

### Effect on Activation Distribution

Without BN:

```
Layer outputs drift over time → unstable training
```

With BN:

```
Layer outputs stay normalized → smooth gradient flow
```

---

### Variants of Normalization

| Method       | Normalization Scope     | Typical Use      |
| ------------ | ----------------------- | ---------------- |
| BatchNorm    | across batch            | CNNs, MLPs       |
| LayerNorm    | across features         | Transformers     |
| InstanceNorm | per sample, per channel | Style transfer   |
| GroupNorm    | groups of channels      | small-batch CNNs |
| RMSNorm      | root mean square        | LLMs             |

---

### When BatchNorm Works Poorly

| Scenario              | Issue                  |
| --------------------- | ---------------------- |
| Very small batch size | unreliable statistics  |
| RNNs / Transformers   | sequence dependency    |
| Online learning       | unstable running stats |

In such cases, **LayerNorm** or **GroupNorm** is preferred.

---

### Summary

Batch Normalization reshapes the optimization landscape by stabilizing activation distributions, enabling deeper networks, faster convergence, and more reliable training.
It is not merely a normalization trick, but a fundamental optimization tool in deep learning.

